# blockwise-ilastik-multicut

This notebook walks through the full pipeline:
1. Inspect an ilastik `.ilp` project file
2. Re-fit a sklearn RandomForestClassifier on the stored training data
3. Run blockwise multicut on a new (large) volume

See `README.md` for requirements and background.

In [ ]:
# --- Paths — edit these ---
ILP_PATH = "/path/to/your_project.ilp"
RF_SAVE_PATH = "rf.pkl"

# Channel files: map the channel names stored in the .ilp file to actual data.
# Run the 'Inspect feature names' cell below first to see what channel names
# your project uses.
CHANNEL_FILES = {
    "Membrane Probabilities 0": ("/path/to/boundary.h5", "/data"),
    "Raw Data 0":               ("/path/to/raw.h5",      "/data"),
}

OUTPUT_H5   = "segmentation.h5"
OUTPUT_KEY  = "/seg"
BETA        = 0.5      # boundary bias: <0.5 favours merging, >0.5 splitting
BLOCK_SHAPE = (256, 256, 256)
HALO        = (32, 32, 32)
N_THREADS   = 8
USE_2DWS    = False    # set True for strongly anisotropic (2D-slice) data

## 1. Inspect the .ilp file

In [ ]:
from ilp_reader import read_feature_names, read_edge_labels, read_edge_features

feature_names = read_feature_names(ILP_PATH)
print("Channels and selected features stored in the .ilp file:")
for channel, feats in feature_names.items():
    print(f"  {channel!r}: {feats}")

In [ ]:
labels_dict = read_edge_labels(ILP_PATH)
import numpy as np
labels_arr = np.array(list(labels_dict.values()), dtype=np.uint8)
print(f"{len(labels_arr)} annotated edges")
for lbl, name in [(1, "merge"), (2, "split")]:
    print(f"  {name} (label={lbl}): {(labels_arr == lbl).sum()}")

In [ ]:
# Preview the cached edge feature DataFrame
edge_features_df = read_edge_features(ILP_PATH)
print(f"EdgeFeatures shape: {edge_features_df.shape}")
edge_features_df.head()

## 2. Re-fit sklearn RF on the training data

In [ ]:
from fit_classifier import fit_rf_from_ilp
import pickle

rf = fit_rf_from_ilp(
    ilp_path=ILP_PATH,
    n_estimators=200,
    n_jobs=N_THREADS,
    random_state=42,
    verbose=True,
)

with open(RF_SAVE_PATH, "wb") as f:
    pickle.dump(rf, f)
print(f"Saved classifier to {RF_SAVE_PATH}")

In [ ]:
# --- Feature importances ---
import pandas as pd
from ilp_reader import read_training_data

_, _, feature_cols = read_training_data(ILP_PATH)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(15).plot.barh(figsize=(8, 5), title="Top-15 feature importances");

## 3. Run blockwise multicut on new data

In [ ]:
from multicut_from_ilp import run_blockwise_multicut

# Convert CHANNEL_FILES dict to the spec format expected by run_blockwise_multicut
channel_specs = [
    f"{name}:{path}:{key}" for name, (path, key) in CHANNEL_FILES.items()
]

seg = run_blockwise_multicut(
    ilp_path=ILP_PATH,
    rf_path=RF_SAVE_PATH,
    channel_specs=channel_specs,
    output_path=OUTPUT_H5,
    output_key=OUTPUT_KEY,
    beta=BETA,
    block_shape=BLOCK_SHAPE,
    halo=HALO,
    n_threads=N_THREADS,
    use_2dws=USE_2DWS,
    verbose=True,
)

In [ ]:
# --- Quick visual check (middle slice) ---
import matplotlib.pyplot as plt
import h5py, numpy as np

with h5py.File(CHANNEL_FILES[next(iter(CHANNEL_FILES))][0], "r") as f:
    raw = f[next(iter(CHANNEL_FILES.values()))[1]][()]

z = raw.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(raw[z], cmap="gray"); axes[0].set_title("Raw (middle slice)")
axes[1].imshow(seg[z], cmap="tab20"); axes[1].set_title("Segmentation")
plt.tight_layout()